# 02 — Mô hình 1 (TextCNN) và Mô hình 2 (BiLSTM)

Hai mô hình "from scratch" theo yêu cầu đề bài:

| # | Họ | Kiến trúc |
|---|---|---|
| 1 | CNN-based | TextCNN: Conv1D đa kernel + max-pool cho mỗi câu, ghép `[u; v; abs(u-v); u*v]` |
| 2 | RNN/LSTM/GRU | BiLSTM chia sẻ trọng số + attention pooling, cùng công thức ghép |

Notebook này **không định nghĩa lại kiến trúc hay vòng lặp train** — mọi thứ nằm trong
`src/models.py` và `src/trainer.py` để đảm bảo notebook và `python src/train.py` cho ra
cùng một kết quả (yêu cầu tái lập ở Phụ lục A).

Chạy `01_eda.ipynb` trước để có `data/splits/*.csv`.

## Setup Kaggle — kéo code từ GitHub

Chạy cell này **trước tiên** trên Kaggle. Bỏ qua được khi chạy ở máy local.
Sửa code ở máy → `git push` → chạy lại cell này để lấy bản mới.

In [ ]:
import os

REPO_URL = "https://github.com/dofu18/ViANLI_DL_NLP.git"

if os.path.exists("/kaggle/input"):
    !rm -rf /kaggle/working/repo
    !git clone -q $REPO_URL /kaggle/working/repo
    !cp -r /kaggle/working/repo/src /kaggle/working/
    !cp -r /kaggle/working/repo/configs /kaggle/working/
    !cp -r /kaggle/working/repo/data /kaggle/working/     # split cố định từ 01_eda
    print("src/:", sorted(os.listdir("/kaggle/working/src")))
else:
    print("Chạy local — bỏ qua bước clone.")

In [ ]:
# !pip install -q underthesea

import json, os, sys, time

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

ON_KAGGLE = os.path.exists("/kaggle/input")
ROOT = "/kaggle/working" if ON_KAGGLE else os.path.abspath("..")
sys.path.insert(0, os.path.join(ROOT, "src"))

FIG_DIR = os.path.join(ROOT, "outputs", "figures")
LOG_DIR = os.path.join(ROOT, "outputs", "logs")
CKPT_DIR = os.path.join(ROOT, "outputs", "checkpoints")
for d in (FIG_DIR, LOG_DIR, CKPT_DIR):
    os.makedirs(d, exist_ok=True)

from data import (LABELS, Vocab, build_embedding_matrix, load_splits,
                  make_loaders, set_seed)
from models import BiLSTMNLI, TextCNNNLI, count_params
from trainer import predict, train_model
from evaluate import (error_examples, full_report, plot_confusion_matrix,
                      plot_learning_curve)

SEED = 42
set_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device =", DEVICE, "|", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")

## 1. Nạp split cố định

Ưu tiên `data/splits/*.csv` do `01_eda.ipynb` sinh ra — đảm bảo cả 4 mô hình dùng
đúng cùng một phân chia. Nếu chưa có thì gọi lại `load_splits()`.

In [ ]:
SPLIT_DIR = os.path.join(ROOT, "data", "splits")
COLS = {"premise": "premise", "hypothesis": "hypothesis", "label": "label"}

def load_from_csv():
    ds = {}
    for name in ("train", "validation", "test"):
        df = pd.read_csv(os.path.join(SPLIT_DIR, f"{name}.csv"), encoding="utf-8")
        df["premise"] = df["premise"].fillna("")
        df["hypothesis"] = df["hypothesis"].fillna("")
        # dict-of-list: khớp interface mà NLIPairDataset mong đợi
        ds[name] = {c: df[c].tolist() for c in ("premise", "hypothesis", "label")}
    return ds, COLS

if os.path.exists(os.path.join(SPLIT_DIR, "train.csv")):
    ds, cols = load_from_csv()
    print("Nạp từ data/splits/ (split cố định)")
else:
    hf_ds, cols = load_splits(seed=SEED)
    ds = {k: {c: list(v[cols[c]]) for c in cols} for k, v in hf_ds.items()}
    cols = COLS
    print("CẢNH BÁO: chưa có data/splits/ — chạy 01_eda.ipynb trước để cố định split")

{k: len(v["label"]) for k, v in ds.items()}

## 2. Vocab — chỉ xây từ TRAIN

Xây vocab từ dev/test là một dạng rò rỉ. `Vocab.build()` chỉ nhận văn bản của train.

In [ ]:
SEGMENT = True     # tách từ tiếng Việt bằng underthesea
MIN_FREQ = 2
MAX_LEN = 128      # chốt lại theo p95 trong 01_eda.ipynb
EMB_DIM = 300

t0 = time.time()
vocab = Vocab.build(ds["train"]["premise"] + ds["train"]["hypothesis"],
                    min_freq=MIN_FREQ, segment=SEGMENT)
vocab.save(os.path.join(CKPT_DIR, "shared_vocab.txt"))
print(f"|V| = {len(vocab)}  ({time.time() - t0:.1f}s)")
print("20 token đầu:", vocab.itos[:20])

## 3. Embedding PhoW2V

Tải [PhoW2V 300 chiều](https://github.com/datquocnguyen/PhoW2V) và upload thành Kaggle
Dataset, hoặc đặt `EMB_PATH = None` để chạy nhánh random init (nhánh ablation).

In [ ]:
EMB_PATH = None   # ví dụ: "/kaggle/input/phow2v/word2vec_vi_words_300dims.txt"

if EMB_PATH and os.path.exists(EMB_PATH):
    emb_matrix = build_embedding_matrix(vocab, EMB_PATH, EMB_DIM)
else:
    emb_matrix = None
    print("Không có PhoW2V -> khởi tạo embedding ngẫu nhiên. "
          "Ghi rõ điều này trong báo cáo (mục ablation).")

## 4. Hàm chạy một thí nghiệm

Dùng chung cho cả TextCNN, BiLSTM và mọi nhánh ablation — tránh lệch giao thức.

In [ ]:
BASE_CFG = {
    "seed": SEED,
    "epochs": 30,
    "batch_size": 64,
    "learning_rate": 1e-3,
    "weight_decay": 0.0,
    "clip_grad": 5.0,
    "early_stopping_patience": 5,
    "class_weights": None,   # bật nếu 01_eda cho thấy nhãn lệch mạnh
}

RESULTS = {}

def run(run_name, arch, *, dropout=0.5, hypothesis_only=False,
        pretrained_emb=emb_matrix, max_len=MAX_LEN, **overrides):
    """Train + đánh giá một cấu hình; lưu log/hình/checkpoint và trả về summary."""
    set_seed(SEED)   # reset seed trước MỖI thí nghiệm để so sánh công bằng
    cfg = {**BASE_CFG, **overrides, "run_name": run_name, "arch": arch,
           "dropout": dropout, "max_length": max_len,
           "hypothesis_only": hypothesis_only,
           "pretrained_emb": "phow2v" if pretrained_emb is not None else "none"}

    common = dict(vocab_size=len(vocab), emb_dim=EMB_DIM, dropout=dropout,
                  pretrained_emb=pretrained_emb)
    if arch == "textcnn":
        model = TextCNNNLI(n_filters=cfg.get("n_filters", 128),
                           kernel_sizes=tuple(cfg.get("kernel_sizes", (2, 3, 4, 5))),
                           **common)
    elif arch == "bilstm":
        model = BiLSTMNLI(hidden=cfg.get("hidden", 256),
                          num_layers=cfg.get("num_layers", 1), **common)
    else:
        raise ValueError(arch)

    loaders = make_loaders(ds, cols, vocab, batch_size=cfg["batch_size"],
                           max_len=max_len, segment=SEGMENT,
                           hypothesis_only=hypothesis_only,
                           num_workers=2 if ON_KAGGLE else 0)

    print(f"\n=== {run_name} | {arch} | {count_params(model):,} tham số ===")
    started = time.time()
    model, history = train_model(
        model, loaders, cfg, device=DEVICE,
        log_path=os.path.join(LOG_DIR, f"{run_name}_history.json"))
    elapsed = time.time() - started

    y_true, y_pred = predict(model, loaders["test"], DEVICE)
    report = full_report(y_true, y_pred,
                         out_json=os.path.join(LOG_DIR, f"{run_name}_test.json"))
    plot_learning_curve(history, run_name,
                        os.path.join(FIG_DIR, f"{run_name}_curve.png"))
    plot_confusion_matrix(y_true, y_pred, run_name,
                          os.path.join(FIG_DIR, f"{run_name}_cm.png"))
    os.makedirs(os.path.join(CKPT_DIR, run_name), exist_ok=True)
    torch.save(model.state_dict(), os.path.join(CKPT_DIR, run_name, "best.pt"))

    summary = {"run_name": run_name, "arch": arch,
               "accuracy": report["accuracy"], "macro_f1": report["macro_f1"],
               "weighted_f1": report["weighted_f1"],
               "params": count_params(model),
               "epochs_chạy": len(history),
               "train_seconds": round(elapsed, 1)}
    with open(os.path.join(LOG_DIR, f"{run_name}_summary.json"), "w",
              encoding="utf-8") as f:
        json.dump({**summary, "config": cfg}, f, ensure_ascii=False, indent=2)
    RESULTS[run_name] = {**summary, "_preds": (y_true, y_pred), "_history": history}
    print(json.dumps(summary, ensure_ascii=False, indent=2))
    return summary

## 5. Mô hình 1 — TextCNN

In [ ]:
run("textcnn", "textcnn")

## 6. Mô hình 2 — BiLSTM + attention

In [ ]:
run("bilstm_attn", "bilstm")

## 7. Ablation

Đề bài yêu cầu phân tích **ít nhất một** yếu tố; ở đây chạy bốn nhánh trên cùng
kiến trúc BiLSTM (rẻ, nhanh) để bảng ablation có chiều sâu.

| Nhánh | Câu hỏi |
|---|---|
| `no_pretrained_emb` | PhoW2V đóng góp bao nhiêu? |
| `hypothesis_only` | Mô hình đoán được nhãn khi **không** nhìn premise? → đo bias |
| `dropout_0.2` | Regularization ảnh hưởng thế nào trên bộ adversarial? |
| `maxlen_64` | Cắt ngắn đầu vào có mất thông tin không? |

Bỏ bớt nhánh nếu hết quota GPU — nhưng phải **ghi rõ trong báo cáo** nhánh nào đã bỏ.

In [ ]:
if emb_matrix is not None:
    run("bilstm_no_pretrained_emb", "bilstm", pretrained_emb=None)
else:
    print("Bỏ qua: baseline vốn đã dùng embedding ngẫu nhiên.")

In [ ]:
run("bilstm_hyponly", "bilstm", hypothesis_only=True)

In [ ]:
run("bilstm_dropout02", "bilstm", dropout=0.2)
run("bilstm_maxlen64", "bilstm", max_len=64)

## 8. Tổng hợp

In [ ]:
table = pd.DataFrame([
    {k: v for k, v in r.items() if not k.startswith("_")}
    for r in RESULTS.values()
]).sort_values("accuracy", ascending=False)
display(table.round(4))
table.to_csv(os.path.join(LOG_DIR, "cnn_rnn_results.csv"), index=False)

In [ ]:
# Lưu dự đoán trên test để 05_analysis.ipynb phân tích lỗi chéo giữa các mô hình
pred_dir = os.path.join(LOG_DIR, "predictions")
os.makedirs(pred_dir, exist_ok=True)
for name, r in RESULTS.items():
    np.save(os.path.join(pred_dir, f"{name}.npy"), r["_preds"][1])
np.save(os.path.join(pred_dir, "y_true.npy"), RESULTS[list(RESULTS)[0]]["_preds"][0])
print(sorted(os.listdir(pred_dir)))

In [ ]:
# Learning curve của 2 mô hình chính đặt chung một trục để so sánh
fig, ax = plt.subplots(figsize=(6, 4))
for name in ("textcnn", "bilstm_attn"):
    if name in RESULTS:
        h = RESULTS[name]["_history"]
        ax.plot([r["epoch"] for r in h], [r["val_macro_f1"] for r in h],
                marker="o", label=name)
ax.set_xlabel("Epoch"); ax.set_ylabel("Val macro-F1")
ax.set_title("TextCNN vs BiLSTM trên dev")
ax.legend(); plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "cnn_vs_rnn_dev.png"), dpi=150)
plt.show()

## 9. Xem nhanh vài lỗi

Phân tích lỗi đầy đủ nằm ở `05_analysis.ipynb`; đây chỉ là kiểm tra nhanh xem mô hình
có sập về dự đoán một nhãn duy nhất hay không.

In [ ]:
best = table.iloc[0]["run_name"]
y_true, y_pred = RESULTS[best]["_preds"]
print(f"Mô hình tốt nhất: {best}")
print("Phân bố dự đoán:",
      {LABELS[i]: int((y_pred == i).sum()) for i in range(len(LABELS))})

pd.set_option("display.max_colwidth", 100)
pd.DataFrame(error_examples(ds["test"], cols, y_true, y_pred, n=10))

## Cần điền vào báo cáo

- Bảng cấu hình mô hình 1 & 2 (kiến trúc, số tham số, regularization, loss).
- Bảng siêu tham số (batch, epoch, lr, optimizer, scheduler, early stopping).
- Hình: `textcnn_curve.png`, `bilstm_attn_curve.png`, `cnn_vs_rnn_dev.png`, hai confusion matrix.
- Bảng ablation từ `cnn_rnn_results.csv`.

**Cảnh báo diễn giải:** ViANLI là bộ adversarial. Nếu TextCNN/BiLSTM chỉ nhỉnh hơn
majority baseline vài điểm, đó là kết quả hợp lệ — hai kiến trúc này không có cross-attention
giữa premise và hypothesis nên gần như không suy luận được. Hãy trình bày và giải thích,
đừng cố tinh chỉnh để "đẹp số".